In [23]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple
import torch.nn.functional as F

In [36]:
AUDIO_DIR  = "baby_data/audio" #"../../data/clean/audio/vocals/0.mp3"
LABEL_DIR = "baby_data/labels" #"../../data/clean/subtitles/0.csv"

In [41]:
class JapaneseSyllableProcessor:
    def __init__(self,
                 sample_rate: int = 16000,
                 frame_length: float = 0.025,  # 25ms frames
                 frame_shift: float = 0.010):  # 10ms shift
        """
        Processor for Japanese syllable-level audio analysis

        Args:
            sample_rate: Target audio sample rate
            frame_length: Analysis frame length in seconds
            frame_shift: Frame shift in seconds
        """
        self.sample_rate = sample_rate
        self.frame_length = frame_length
        self.frame_shift = frame_shift

        # Basic hiragana syllable set (can be customized)
        self.basic_syllables = {
            'あ', 'い', 'う', 'え', 'お',
            'か', 'き', 'く', 'け', 'こ',
            'さ', 'し', 'す', 'せ', 'そ',
            'た', 'ち', 'つ', 'て', 'と',
            'な', 'に', 'ぬ', 'ね', 'の',
            'は', 'ひ', 'ふ', 'へ', 'ほ',
            'ま', 'み', 'む', 'め', 'も',
            'や', 'ゆ', 'よ',
            'ら', 'り', 'る', 'れ', 'ろ',
            'わ', 'を', 'ん',
            'が', 'ぎ', 'ぐ', 'げ', 'ご',
            'ざ', 'じ', 'ず', 'ぜ', 'ぞ',
            'だ', 'ぢ', 'づ', 'で', 'ど',
            'ば', 'び', 'ぶ', 'べ', 'ぼ',
            'ぱ', 'ぴ', 'ぷ', 'ぺ', 'ぽ',
            'きゃ', 'きゅ', 'きょ',
            'しゃ', 'しゅ', 'しょ',
            'ちゃ', 'ちゅ', 'ちょ',
            'にゃ', 'にゅ', 'にょ',
            'ひゃ', 'ひゅ', 'ひょ',
            'みゃ', 'みゅ', 'みょ',
            'りゃ', 'りゅ', 'りょ',
            'ぎゃ', 'ぎゅ', 'ぎょ',
            'じゃ', 'じゅ', 'じょ',
            'びゃ', 'びゅ', 'びょ',
            'ぴゃ', 'ぴゅ', 'ぴょ',
            'でぃ', 'ふぁ', 'ふぃ', 'ふぇ', 'ふぉ',
            '<silence>'
        }

        # Syllable to index mapping
        self.syllable_to_idx = {syl: idx for idx, syl in enumerate(self.basic_syllables)}
        self.idx_to_syllable = {idx: syl for syl, idx in self.syllable_to_idx.items()}

        # Audio preprocessing transforms
        self.audio_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_mels=80,
            n_fft=int(frame_length * sample_rate),
            hop_length=int(frame_shift * sample_rate)
        )

        self.normalize = torchaudio.transforms.AmplitudeToDB()

    def load_and_preprocess_audio(self,
                                  audio_path: str,
                                  start_time: float = None,
                                  end_time: float = None) -> torch.Tensor:
        """
        Load and preprocess audio segment

        Args:
            audio_path: Path to MP3 file
            start_time: Segment start time in seconds
            end_time: Segment end time in seconds

        Returns:
            Preprocessed audio features
        """
        # Load audio
        waveform, orig_sr = torchaudio.load(audio_path)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample if needed
        if orig_sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_sr, self.sample_rate)
            waveform = resampler(waveform)

        # Extract segment if times provided
        if start_time is not None and end_time is not None:
            start_frame = int(start_time * self.sample_rate)
            end_frame = int(end_time * self.sample_rate)
            waveform = waveform[:, start_frame:end_frame]

        # Convert to mel spectrogram
        mel_spec = self.audio_transform(waveform)

        # Apply log-scale and normalization
        mel_spec = self.normalize(mel_spec)

        return mel_spec


class JapaneseSyllableDataset:
    def __init__(self,
                 audio_dir: str,
                 annotation_dir: str,
                 processor: JapaneseSyllableProcessor):
        """
        Dataset for Japanese syllable-level analysis

        Args:
            audio_dir: Directory containing MP3 files
            annotation_dir: Directory containing CSV files with timing annotations
            processor: Syllable processor instance
        """
        self.audio_dir = Path(audio_dir)
        self.annotation_dir = Path(annotation_dir)
        self.processor = processor

        # Get all audio and annotation files
        self.audio_files = sorted(list(self.audio_dir.glob('*.mp3')))
        self.annotation_files = sorted(list(self.annotation_dir.glob('*.csv')))

        # Validate matching files
        if len(self.audio_files) != len(self.annotation_files):
            raise ValueError("Number of audio files does not match number of annotation files")

        # Load and process all annotations
        self.annotations = self._load_all_annotations()

    def _load_all_annotations(self) -> pd.DataFrame:
        """
        Load and preprocess all annotation CSVs

        Returns:
            Concatenated DataFrame with all annotations
        """
        all_annotations = []

        for audio_file, annotation_file in zip(self.audio_files, self.annotation_files):
            # Load single CSV
            df = pd.read_csv(annotation_file)

            # Convert times to float if needed
            df['start'] = pd.to_numeric(df['start'])
            df['end'] = pd.to_numeric(df['end'])

            # Add audio file path column
            df['audio_path'] = str(audio_file)

            # Filter for valid syllables
            df = df[df['token'].isin(self.processor.basic_syllables)]

            all_annotations.append(df)

        # Concatenate all annotations
        return pd.concat(all_annotations, ignore_index=True)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        """
        Get a single syllable segment with its features
        
        Returns:
            Dict containing audio features and syllable info
        """
        row = self.annotations.iloc[idx]

        # Extract audio segment using the file path stored in annotations
        features = self.processor.load_and_preprocess_audio(
            row['audio_path'],
            row['start'],
            row['end']
        )

        # Ensure features are 2D [channels, time]
        if features.dim() == 3:
            features = features.squeeze(0)

        # Convert syllable to index
        syllable_idx = self.processor.syllable_to_idx[row['token']]

        return {
            'features': features,
            'syllable_idx': torch.tensor(syllable_idx, dtype=torch.long),
            'start': row['start'],
            'end': row['end'],
            'audio_path': row['audio_path']
        }


def collate_variable_length(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    # Get max length in batch
    max_length = max(s['features'].shape[-1] for s in batch)

    # Pad features to max length
    features = []
    for s in batch:
        feat = s['features']
        
        print(feat.dim())
        raise(Exception("stop"))

        # Ensure 3D tensor [channels, time]
        if feat.dim() == 2:
            feat = feat.unsqueeze(0)  # Add channel dimension if missing

        # Pad to max length
        if feat.shape[-1] < max_length:
            pad_width = max_length - feat.shape[-1]
            feat = F.pad(feat, (0, pad_width))

        features.append(feat)

    # Stack features
    features_tensor = torch.stack(features)

    # Ensure final shape is [batch, time, features]
    features_tensor = features_tensor.permute(0, 2, 1)

    return {
        'features': features_tensor,
        'syllable_idx': torch.stack([s['syllable_idx'] for s in batch]),
        'starts': torch.tensor([s['start'] for s in batch]),
        'ends': torch.tensor([s['end'] for s in batch])
    }

def prepare_training_data(audio_dir: str,
                          annotation_dir: str,
                          batch_size: int = 32) -> torch.utils.data.DataLoader:
    """
    Prepare data loader for training
    
    Args:
        audio_dir: Directory containing MP3 files
        annotation_dir: Directory containing timing annotations
        batch_size: Training batch size
        
    Returns:
        DataLoader instance
    """
    processor = JapaneseSyllableProcessor()
    dataset = JapaneseSyllableDataset(audio_dir, annotation_dir, processor)
    
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_variable_length,
        num_workers=4,
        pin_memory=True
    )

In [42]:
# see size of batch
train_loader = prepare_training_data(AUDIO_DIR, LABEL_DIR, batch_size=64)
next(iter(train_loader))['features'].shape

2


Exception: Caught Exception in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
  File "/tmp/ipykernel_241027/1546583647.py", line 207, in collate_variable_length
    raise(Exception("stop"))
Exception: stop


In [30]:
class SyllableAlignmentModel(nn.Module):
    def __init__(self,
                 input_dim: int = 80,  # Mel spectrogram features
                 hidden_dim: int = 256,
                 num_syllables: int = 45,  # Based on hiragana syllable set
                 num_layers: int = 3):
        super().__init__()

        # Feature encoder with adjusted input handling
        self.feature_encoder = nn.Sequential(
            # Transpose and reshape to match conv1d expectations
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
        )

        # Rest of the code remains the same...

    def forward(self,
                x: torch.Tensor,
                syllable_targets: torch.Tensor = None) -> Tuple[torch.Tensor, torch.Tensor]:
        # Handle potential input variations
        if x.dim() == 4:
            x = x.squeeze(1)  # Remove singleton dimension

        # Ensure correct dimensions: [batch, time, features]
        if x.dim() == 3:
            # Transpose to [batch, features, time] for Conv1d
            x = x.transpose(1, 2)

        # Feature encoding
        features = self.feature_encoder(x)

        # Transpose back to [batch, time, features]
        features = features.transpose(1, 2)

        # Transformer encoding
        encoded = self.transformer_encoder(features)

        # Global average pooling
        global_repr = encoded.mean(dim=1)

        # Projection
        projected_repr = F.normalize(self.projection_head(global_repr), dim=-1)

        # Syllable classification (optional)
        if syllable_targets is not None:
            classification_loss = F.cross_entropy(
                self.syllable_classifier(global_repr),
                syllable_targets
            )
        else:
            classification_loss = torch.tensor(0.0).to(x.device)

        return classification_loss, projected_repr


In [31]:
class ContrastiveLearningTrainer:
    def __init__(self,
                 model: nn.Module,
                 optimizer: torch.optim.Optimizer,
                 device: str = 'cuda'):
        """
        Contrastive learning trainer
        
        Args:
            model: Syllable alignment model
            optimizer: Training optimizer
            device: Computing device
        """
        self.model = model.to(device)
        self.optimizer = optimizer
        self.device = device

    def compute_contrastive_loss(self,
                                 representations: torch.Tensor,
                                 temperature: float = 0.1) -> torch.Tensor:
        """
        Compute NT-Xent (Normalized Temperature Cross Entropy) loss
        
        Args:
            representations: Normalized representations
            temperature: Temperature scaling parameter
        
        Returns:
            Contrastive loss
        """
        # Compute similarity matrix
        similarity_matrix = torch.matmul(representations, representations.T)

        # Mask out diagonal (self-similarities)
        batch_size = representations.shape[0]
        mask = torch.eye(batch_size, dtype=torch.bool).to(self.device)
        similarity_matrix.masked_fill_(mask, float('-inf'))

        # Compute softmax similarities
        similarities = F.softmax(similarity_matrix / temperature, dim=1)

        # Compute loss
        positive_pairs = torch.sum(torch.log(similarities + 1e-10), dim=1)
        loss = -torch.mean(positive_pairs)

        return loss

    def train_epoch(self,
                    dataloader: torch.utils.data.DataLoader,
                    epoch: int) -> Dict[str, float]:
        """
        Train for one epoch
        
        Args:
            dataloader: Training data loader
            epoch: Current epoch number
        
        Returns:
            Dictionary of training metrics
        """
        self.model.train()
        total_loss = 0
        total_classification_loss = 0
        total_contrastive_loss = 0

        for batch_idx, batch in enumerate(dataloader):
            # Move to device
            features = batch['features'].to(self.device)
            syllable_idx = batch['syllable_idx'].to(self.device)

            # Zero gradients
            self.optimizer.zero_grad()

            # Forward pass
            classification_loss, representations = self.model(
                features,
                syllable_targets=syllable_idx
            )

            # Compute contrastive loss
            contrastive_loss = self.compute_contrastive_loss(
                representations,
                temperature=self.model.temperature
            )

            # Combined loss
            loss = classification_loss + contrastive_loss

            # Backward pass
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            # Optimizer step
            self.optimizer.step()

            # Logging
            total_loss += loss.item()
            total_classification_loss += classification_loss.item()
            total_contrastive_loss += contrastive_loss.item()

            # Optional: Print progress
            if batch_idx % 50 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}: "
                      f"Total Loss {loss.item():.4f}, "
                      f"Classification Loss {classification_loss.item():.4f}, "
                      f"Contrastive Loss {contrastive_loss.item():.4f}")

        # Compute average metrics
        avg_loss = total_loss / len(dataloader)
        avg_classification_loss = total_classification_loss / len(dataloader)
        avg_contrastive_loss = total_contrastive_loss / len(dataloader)

        return {
            'total_loss': avg_loss,
            'classification_loss': avg_classification_loss,
            'contrastive_loss': avg_contrastive_loss
        }

    def train(self,
              train_loader: torch.utils.data.DataLoader,
              num_epochs: int = 50,
              validation_loader: torch.utils.data.DataLoader = None):
        """
        Full training procedure
        
        Args:
            train_loader: Training data loader
            num_epochs: Number of training epochs
            validation_loader: Optional validation data loader
        """
        best_loss = float('inf')

        for epoch in range(1, num_epochs + 1):
            # Training phase
            train_metrics = self.train_epoch(train_loader, epoch)

            # Optional validation
            if validation_loader:
                val_metrics = self.validate(validation_loader)
                print(f"Validation Metrics: {val_metrics}")

            # Model checkpointing
            if train_metrics['total_loss'] < best_loss:
                best_loss = train_metrics['total_loss']
                self.save_checkpoint(epoch, train_metrics)

            # Learning rate scheduling (optional)
            # self.scheduler.step()

    def validate(self,
                 validation_loader: torch.utils.data.DataLoader) -> Dict[str, float]:
        """
        Validation procedure
        
        Args:
            validation_loader: Validation data loader
        
        Returns:
            Validation metrics
        """
        self.model.eval()
        total_loss = 0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for batch in validation_loader:
                features = batch['features'].to(self.device)
                syllable_idx = batch['syllable_idx'].to(self.device)

                # Forward pass
                classification_loss, representations = self.model(
                    features,
                    syllable_targets=syllable_idx
                )

                # Compute accuracy
                predictions = self.model.syllable_classifier(representations)
                _, predicted = torch.max(predictions, 1)
                total_correct += (predicted == syllable_idx).sum().item()
                total_samples += syllable_idx.size(0)

                total_loss += classification_loss.item()

        # Compute metrics
        avg_loss = total_loss / len(validation_loader)
        accuracy = total_correct / total_samples

        return {
            'validation_loss': avg_loss,
            'accuracy': accuracy
        }

    def save_checkpoint(self,
                        epoch: int,
                        metrics: Dict[str, float],
                        filename: str = 'best_model.pth'):
        """
        Save model checkpoint
        
        Args:
            epoch: Current epoch
            metrics: Training metrics
            filename: Checkpoint filename
        """
        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metrics': metrics
        }, filename)

# Example training script
def main():
    # Prepare data
    audio_path = AUDIO_DIR
    annotation_csv = LABEL_DIR

    # Create data loaders
    train_loader = prepare_training_data(audio_path, annotation_csv, batch_size=64)

    # Split into train/validation if needed
    # validation_loader = prepare_validation_data(...)

    processor = JapaneseSyllableProcessor()

    # Initialize model
    model = SyllableAlignmentModel(
        input_dim=80,  # Mel spectrogram features
        hidden_dim=256,
        num_syllables=len(processor.basic_syllables)
    )

    # Optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-5
    )

    # Trainer
    trainer = ContrastiveLearningTrainer(
        model,
        optimizer,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    # Start training
    trainer.train(
        train_loader,
        num_epochs=50,
        # validation_loader=validation_loader  # Optional
    )

if __name__ == "__main__":
    main()


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 351, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/eizigi/.local/share/virtualenvs/CS_541_Project-RTvQqKvc/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
  File "/tmp/ipykernel_241027/3853810365.py", line 204, in collate_variable_length
    features_tensor = features_tensor.permute(0, 2, 1)
RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 4 is not equal to len(dims) = 3


In [3]:
tokens = {'あ', 'い', 'う', 'え', 'お',
          'か', 'き', 'く', 'け', 'こ',
          'さ', 'し', 'す', 'せ', 'そ',
          'た', 'ち', 'つ', 'て', 'と',
          'な', 'に', 'ぬ', 'ね', 'の',
          'は', 'ひ', 'ふ', 'へ', 'ほ',
          'ま', 'み', 'む', 'め', 'も',
          'や', 'ゆ', 'よ',
          'ら', 'り', 'る', 'れ', 'ろ',
          'わ', 'を', 'ん',
          'が', 'ぎ', 'ぐ', 'げ', 'ご',
          'ざ', 'じ', 'ず', 'ぜ', 'ぞ',
          'だ', 'ぢ', 'づ', 'で', 'ど',
          'ば', 'び', 'ぶ', 'べ', 'ぼ',
          'ぱ', 'ぴ', 'ぷ', 'ぺ', 'ぽ',
          'きゃ', 'きゅ', 'きょ',
          'しゃ', 'しゅ', 'しょ',
          'ちゃ', 'ちゅ', 'ちょ',
          'にゃ', 'にゅ', 'にょ',
          'ひゃ', 'ひゅ', 'ひょ',
          'みゃ', 'みゅ', 'みょ',
          'りゃ', 'りゅ', 'りょ',
          'ぎゃ', 'ぎゅ', 'ぎょ',
          'じゃ', 'じゅ', 'じょ',
          'びゃ', 'びゅ', 'びょ',
          'ぴゃ', 'ぴゅ', 'ぴょ',
          'でぃ', 'ふぁ', 'ふぃ', 'ふぇ', 'ふぉ',
          '<silence>'}

print(len(tokens))

110
